# Probe 004 launcher (phase B)
Colab is a compute worker only. This driver kernel NEVER imports the model stack -- `run.py` runs as a child process, so no kernel restart is ever needed.

**One-time setup:** create a fine-grained GitHub PAT scoped to this single repository, Contents: Read and write, with an expiry. In Colab: key icon (Secrets) -> add `SCOUT_RESULTS_PAT` -> enable notebook access. The PAT never appears in this notebook or its output.

Results branch (contract-bound): `results/probe-004-8b68640183ee`

Per session: run all cells top to bottom. After a disconnect, rerun all cells -- run.py resumes from the bundle on Drive, and the transport cell pushes whatever is new.

In [ ]:
PHASE = 'B'
REPO_URL = 'https://github.com/Moroseui/concept-research-scout.git'
PIN_COMMIT = '33c779ad31fa71b43f4e9ebdd97146bdfc7c1ed9'
RESULTS_BRANCH = 'results/probe-004-8b68640183ee'
OUTPUT_DIR = '/content/drive/MyDrive/concept-research-scout-results/004_v2'

In [ ]:
from google.colab import drive, userdata
import os
drive.mount('/content/drive')
GH_PAT = userdata.get('SCOUT_RESULTS_PAT')  # never printed
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')  # inherited by the run.py child; never printed

In [ ]:
!rm -rf /content/scout-repo
!git clone {REPO_URL} /content/scout-repo
%cd /content/scout-repo
!git checkout {PIN_COMMIT}

In [ ]:
!pip install -q -r probes/004/requirements.txt

In [ ]:
!python probes/004/run.py --phase {PHASE} --output-dir {OUTPUT_DIR}

In [ ]:
# E1 transport: mirror the bundle onto the contract-bound results
# branch. The PAT rides in a header, never in argv or output.
import shutil, subprocess, pathlib, base64, datetime
repo = pathlib.Path('/content/scout-repo')
dest = repo / 'probes/004/results_v2'
if dest.exists(): shutil.rmtree(dest)
shutil.copytree(OUTPUT_DIR, dest)
def git(*a, **k):
    r = subprocess.run(['git', *a], cwd=repo, capture_output=True, text=True, **k)
    if r.returncode: raise SystemExit(f'git {a[0]} failed: {r.stderr[-400:]}')
    return r.stdout
git('config', 'user.email', 'colab-runner@scout.local')
git('config', 'user.name', 'scout colab runner')
auth = base64.b64encode(f'x-access-token:{GH_PAT}'.encode()).decode()
hdr = f'http.extraheader=AUTHORIZATION: basic {auth}'
if subprocess.run(['git', '-c', hdr, 'fetch', 'origin', RESULTS_BRANCH], cwd=repo, capture_output=True).returncode == 0:
    git('checkout', '-B', RESULTS_BRANCH, f'origin/{RESULTS_BRANCH}')
else:
    git('checkout', '-B', RESULTS_BRANCH, PIN_COMMIT)
git('add', '-f', 'probes/004/results_v2')
stamp = datetime.datetime.utcnow().isoformat(timespec='seconds')
subprocess.run(['git', 'commit', '-m', f'session results {stamp}Z'], cwd=repo, capture_output=True)
git('-c', hdr, 'push', 'origin', RESULTS_BRANCH)
print('pushed', RESULTS_BRANCH)

When `run.py` reports the study complete, the results-validate workflow on the pushed branch verifies the bundle and opens the record-result PR. Merging that PR is the human gate.